In [1]:
import ee
import geemap

ee.Initialize()

In [2]:
counties = ee.FeatureCollection("FAO/GAUL/2015/level2")

makueni = counties.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "Kenya"),
        ee.Filter.eq("ADM2_NAME", "Makueni")
    )
)

In [3]:
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(makueni)
    .filterDate("2024-01-01", "2024-03-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
)

In [4]:
image = s2.median().clip(makueni)

In [5]:
ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")

In [6]:
Map = geemap.Map(center=[-2.25, 37.9], zoom=9)

ndvi_vis = {
    "min": 0,
    "max": 0.8,
    "palette": ["brown", "yellow", "green"]
}

Map.addLayer(ndvi, ndvi_vis, "NDVI Jan-Mar 2024")
#Map.addLayer(makueni, {"color": "red"}, "Makueni County")
Map

Map(center=[-2.25, 37.9], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [8]:
# Take all NDVI pixels inside Makueni and calculate their average.

mean_ndvi = ndvi.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=makueni.geometry(),
    scale=10,
    maxPixels=1e9
)

print(mean_ndvi.getInfo())

{'NDVI': 0.5423357669234427}
